# **🏋🏽 Wearable Biomechanics - Temporal Sync**


In [ ]:
%load_ext autoreload
%autoreload 2

# Install OpenDatasetStore & this repo in editable mode (Colab/local)
import importlib.util, os, subprocess, sys
if importlib.util.find_spec("open_dataset_store") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/janithcyapa/OpenDatasetStore.git"])
if importlib.util.find_spec("wearable_biomechanics") is None:
    repo_dir = "/content/Wearable-Biomechanics"
    if not os.path.exists(repo_dir):
        subprocess.check_call(["git", "clone", "-b", "cleannjerk_dataset", "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir])

from wearable_biomechanics import init_store, display_table

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
store = init_store(dataset_path=dataset_path)

In [ ]:
# @title Dataset Summary
store.summary()
display_table(store.list_entries(entry_type='experiments'))

## Visual Sync Pipeline (Front ↔ Side)


In [ ]:
# @title Run Single Visual Sync { display-mode: "form" }
from wearable_biomechanics.sync import (
    visual_estimation_pipeline,
    save_visual_estimation_results,
)

target_entry_id = "experiments_0001" # @param {type:"string"}
generate_annotated = True # @param {type:"boolean"}
generate_synced = True # @param {type:"boolean"}
plot = True # @param {type:"boolean"}

front_df, side_df, side_lag, front_ann, side_ann, synced_vid = visual_estimation_pipeline(
    entry_id=target_entry_id,
    store=store,
    generate_annotated=generate_annotated,
    generate_synced=generate_synced,
    plot=plot,
)

if front_df is not None or side_df is not None:
    save_visual_estimation_results(
        entry_id=target_entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_ann,
        side_annotated_path=side_ann,
        synced_video_path=synced_vid,
    )

In [ ]:
# @title Parallel Batch Visual Sync
from wearable_biomechanics.sync import run_batch_visual_estimation

experiments_to_run = [
    f"experiments_{i:04d}" for i in range(23, 38)
]
run_batch_visual_estimation(experiments_to_run, store=store, max_workers=3, plot=False)

## Raw IMU Sync Pipeline (Front Video ↔ 100Hz IMU)


In [ ]:
# @title Run Single Front-to-Raw Sync { display-mode: "form" }
from wearable_biomechanics.sync import sync_front_to_raw_experiment

target_entry_id = "experiments_0037" # @param {type:"string"}
sync_front_to_raw_experiment(target_entry_id, store=store, plot=True)

In [ ]:
# @title Run Batch Front-to-Raw Sync { display-mode: "form" }
from wearable_biomechanics.sync import run_batch_front_to_raw_sync

start_idx = 1 # @param {type:"integer"}
end_idx = 48 # @param {type:"integer"}
run_batch_front_to_raw_sync(start_idx, end_idx, store=store, plot=True)